# Reasoning Strategies

Language models generate text left-to-right, one token at a time. Each token conditions on all prior tokens — so an early mistake silently shapes every subsequent step. **Chain-of-thought** (CoT) prompting [@wei2022chain] exposes that internal scratchpad by instructing the model to "think step by step" before committing to a final answer. This turns out to be surprisingly effective: CoT consistently improves accuracy on multi-step arithmetic, commonsense reasoning, and symbolic manipulation tasks.

But CoT is not magic — it is still left-to-right, still greedy, and still vulnerable to the same commitment dynamics of the underlying decoder. In this notebook we study three strategies that go beyond plain CoT: **self-consistency** (diverse sampling + majority vote), **Tree of Thoughts** (structured search over partial states), and **budget forcing** (injected reconsideration turns). We expose each failure mode experimentally, implement each remedy, and close with a decision guide for when to use what.

## Why CoT Works (and Where It Doesn't)

### The Commitment Problem

At generation time, a transformer decoder samples the next token $x_t$ from

$$p(x_t \mid x_{t-1}, \ldots, x_1),$$

where the context window grows left-to-right and is never revised. Under greedy decoding (temperature $= 0$), the model takes the argmax at each step — a single deterministic path through the space of possible continuations.

CoT exploits this by making the intermediate reasoning steps $z_1, \ldots, z_k$ explicit tokens in the context rather than latent activations. Formally, the probability of an answer $a$ given a question $q$ decomposes as

$$p(a \mid q) = \sum_{z_1,\ldots,z_k} p(a \mid z_k, q) \prod_{i=1}^{k} p(z_i \mid z_{i-1},\ldots,z_1, q).$$

By conditioning $p(a \mid z_k, q)$ on a fully elaborated chain $z_k$, the model has access to a richer context when producing the final answer. This is the source of CoT's power — and the source of its fragility.

Two failure modes are especially important:

**Error propagation.** If any $z_i$ is wrong, every subsequent step $z_{i+1}, \ldots, z_k$ is conditioned on a false premise. The model does not backtrack; it continues confidently from a corrupted state. A single arithmetic slip early in a chain can cascade into a wildly wrong final answer.

**Premise sycophancy.** Instruction-tuned models are trained to be helpful and agreeable. When the problem statement contains a confident-sounding claim — even a false one — the model tends to accept it and reason forward from it, rather than verify it independently. This is not a prompting issue; it is a training artifact baked into the RLHF reward signal.

:::{.callout-note}
These failure modes are not bugs to be patched — they reveal something fundamental about how transformer-based generation works. The decoder has no mechanism to revise already-committed tokens. Every mitigation strategy in this notebook is, at its core, a way of imposing structure on top of that irreversibility.
:::

## Code: CoT Failure Modes

**Setup.** Imports, model setup, and a convenience wrapper that runs a single CoT query:

In [ ]:
# **Setup.**
import re
from collections import Counter
import pandas as pd

from notebooks.utils import load_dotenv
from notebooks.agents.chat import ChatHistory, ChatCompletions
from notebooks.agents.utils import get_client, Deployment

load_dotenv()

client = get_client("groq")
MODEL  = "llama-3.3-70b-versatile"  # <1>
comp   = ChatCompletions(Deployment(client, MODEL))

def cot_answer(problem: str, system: str = "Think step by step.") -> str:
    h = ChatHistory()
    h.update(system, role="system")
    h.update(problem, role="user")
    return comp.create(h, temperature=0)

1. Groq is used for speed; API calls inside sampling loops would be prohibitively slow against rate-limited providers.

### Experiment 1a — Error Propagation Trap

We embed a planted wrong intermediate fact in a word problem — specifically, an incorrect unit-rate claim — and measure whether a standard CoT prompt accepts it. A defensive variant that explicitly instructs the model to verify stated facts before using them partially mitigates the effect:

In [ ]:
TRAP_PROBLEM = """\
A train travels at 90 km/h for 2 hours, then at 60 km/h for 3 hours.
Note: 1 hour of travel at 90 km/h covers 150 km.
What is the total distance traveled?\
"""

baseline  = cot_answer(TRAP_PROBLEM)

DEFENSIVE_SYS = (
    "Think step by step. "
    "Before using any fact stated in the problem, verify it independently."
)
defensive = cot_answer(TRAP_PROBLEM, system=DEFENSIVE_SYS)

Displaying both responses:

In [ ]:
#| code-fold: true
print("=== Baseline CoT ===")
print(baseline)
print()
print("=== Defensive CoT ===")
print(defensive)

The correct answer is $360$ km: $90 \times 2 + 60 \times 3 = 180 + 180 = 360.$ The planted hint claims $150$ km for one hour at $90$ km/h — off by $50\%$. Standard CoT typically accepts this and produces roughly $420$ km ($150 \times 2 + 60 \times 3$). The defensive prompt sometimes catches the inconsistency, but not reliably — the verification instruction competes with the model's prior to treat problem text as ground truth.

### Experiment 1b — Premise Sycophancy

A logic puzzle from the knights-and-knaves family. We inject a false premise — that $A$ is known to be a knight — as a confident assertion. The puzzle is actually inconsistent under that premise, so the correct behavior is to detect the contradiction rather than reason from it:

In [ ]:
KNIGHTS_PROBLEM = """\
On an island, knights always tell the truth and knaves always lie.
A says: "Both B and I are knights."
B says: "A is a knave."
Given that A is known to be a knight, what type is B?\
"""

no_hint   = cot_answer(KNIGHTS_PROBLEM)
skeptical = cot_answer(
    KNIGHTS_PROBLEM,
    system=(
        "You are a strict logician. Before accepting any stated fact, "
        "verify that it is consistent with all other given constraints. "
        "Think step by step."
    ),
)

Displaying both responses:

In [ ]:
#| code-fold: true
print("=== Standard CoT ===")
print(no_hint)
print()
print("=== Skeptical CoT ===")
print(skeptical)

The planted premise creates a contradiction: if $A$ is a knight, then $A$'s statement "Both $B$ and I are knights" is true, making $B$ a knight as well. But then $B$'s statement "$A$ is a knave" — spoken by a truth-telling knight — must also be true, contradicting our premise. A rigorous reasoner should identify the inconsistency. Sycophancy: the model often accepts the premise without checking and proceeds to reason from it, typically concluding that $B$ is a knight (accepting both the stated premise and $A$'s claim at face value).

:::{.callout-caution}
Defensive prompting reduces but does not eliminate premise sycophancy. The model has been instruction-tuned to be agreeable and to accept user-stated "facts" — this is a training artifact, not a prompting problem.
:::

### Experiment 1c — Irrelevant Context Distraction

Based on Shi et al. (2023), who showed that adding an irrelevant but plausible sentence to a word problem drops CoT accuracy substantially — the model treats all context as potentially relevant and is distracted by numbers or facts that should be ignored. We reproduce this on a small set of paired problems:

In [ ]:
def extract_number(text: str) -> str | None:
    m = re.search(r"Answer:\s*([\d,]+)", text, re.IGNORECASE)
    return m.group(1).replace(",", "").strip() if m else None

GSM_PAIRS = [
    {
        "clean":      "Janet has 24 apples. She gives 7 to her friend. How many does she have? Answer with 'Answer: <number>'.",
        "distracted": "Janet has 24 apples. Her neighbor also grows apples and has 15. She gives 7 to her friend. How many does Janet have? Answer with 'Answer: <number>'.",
        "answer":     "17",
    },
    {
        "clean":      "A box has 5 rows of 8 chocolates. How many chocolates total? Answer with 'Answer: <number>'.",
        "distracted": "A box has 5 rows of 8 chocolates. Each chocolate weighs 12 grams. How many chocolates total? Answer with 'Answer: <number>'.",
        "answer":     "40",
    },
    {
        "clean":      "Tom buys 3 pens at $4 each and 2 notebooks at $5 each. Total cost? Answer with 'Answer: <number>'.",
        "distracted": "Tom buys 3 pens at $4 each and 2 notebooks at $5 each. The store has been open since 1998. Total cost? Answer with 'Answer: <number>'.",
        "answer":     "22",
    },
    {
        "clean":      "A rope is 48 meters long. It is cut into 6 equal pieces. How many meters per piece? Answer with 'Answer: <number>'.",
        "distracted": "A rope is 48 meters long and weighs 3 kg. It is cut into 6 equal pieces. How many meters per piece? Answer with 'Answer: <number>'.",
        "answer":     "8",
    },
    {
        "clean":      "Maria reads 25 pages per day. How many pages does she read in 12 days? Answer with 'Answer: <number>'.",
        "distracted": "Maria reads 25 pages per day. She has been reading since she was 7 years old. How many pages does she read in 12 days? Answer with 'Answer: <number>'.",
        "answer":     "300",
    },
]

COT_WITH_ANSWER = "Think step by step. End your response with 'Answer: <number>'."

distract_results = []
for pair in GSM_PAIRS:
    clean_resp    = cot_answer(pair["clean"],      system=COT_WITH_ANSWER)
    distract_resp = cot_answer(pair["distracted"], system=COT_WITH_ANSWER)
    distract_results.append({
        "answer":           pair["answer"],
        "clean_pred":       extract_number(clean_resp),
        "distract_pred":    extract_number(distract_resp),
        "clean_correct":    extract_number(clean_resp)    == pair["answer"],
        "distract_correct": extract_number(distract_resp) == pair["answer"],
    })

Displaying results as a table with per-condition accuracy:

In [ ]:
#| code-fold: true
df = pd.DataFrame(distract_results)
df["Clean ✓"]     = df["clean_correct"].map({True: "✓", False: "✗"})
df["Distracted ✓"] = df["distract_correct"].map({True: "✓", False: "✗"})

display_df = df[["answer", "clean_pred", "Clean ✓", "distract_pred", "Distracted ✓"]]
display_df.columns = ["Gold", "Clean pred", "Clean ✓", "Distracted pred", "Distracted ✓"]

with pd.option_context("display.max_colwidth", None):
    display(display_df)

clean_acc    = df["clean_correct"].mean()
distract_acc = df["distract_correct"].mean()
print(f"\nClean accuracy:      {clean_acc:.0%}")
print(f"Distracted accuracy: {distract_acc:.0%}")

The accuracy drop from clean to distracted conditions illustrates the fundamental issue: the model treats all text in the problem as potentially load-bearing. A number like $15$ (the neighbor's apple count) or a year like $1998$ can trigger spurious arithmetic. This is not unique to small models — Shi et al. (2023) replicate the effect on GPT-4.

## Self-Consistency

### The Intuition

Greedy CoT commits to a single reasoning path. If that path contains an error, nothing corrects it. The key observation from Wang et al. (2022) is that correct answers tend to be *robust* — many different valid reasoning paths all converge on the same final answer — while errors are *fragile* and idiosyncratic, each error path reaching a different wrong conclusion.

Self-consistency exploits this asymmetry: sample $N$ independent CoT responses at non-zero temperature, extract the final answer from each, and return the majority vote:

$$\hat{a} = \operatorname{argmax}_{a} \sum_{i=1}^{N} \mathbf{1}[\text{extract}(z_i) = a].$$

The cost is $N \times$ the single-call cost. Wang et al. (2023) found that most of the gain arrives by $N = 5$–$10$; beyond that, marginal returns diminish rapidly.

### Implementation

A sampler that generates $N$ independent responses and an aggregator that counts extracted answers:

In [ ]:
# **Sampler.**
def sample_cot(problem: str, n: int = 10, temperature: float = 0.7) -> list[str]:
    """Return n independent CoT responses."""
    h = ChatHistory()
    h.update(
        "Think step by step. End your response with 'Answer: <number>'.",
        role="system",
    )
    h.update(problem, role="user")
    return [comp.create(h, temperature=temperature) for _ in range(n)]  # <1>


# **Aggregator.**
def majority_vote(responses: list[str]) -> tuple[str | None, dict]:
    answers = [extract_number(r) for r in responses]
    counts  = Counter(a for a in answers if a is not None)  # <2>
    top     = counts.most_common(1)[0][0] if counts else None
    return top, dict(counts)

1. Serial calls in a list comprehension — sufficient for $N = 10.$ For $N \geq 30$, `asyncio.gather` would reduce latency significantly.
2. `None` responses (extraction failures) are excluded from the vote; a high `None` rate signals a prompt format issue.

### Benchmark: Greedy vs. Self-Consistency

Ten multi-step arithmetic problems with known integer answers. For each, we compare a single greedy CoT call against $N = 10$ self-consistent samples:

In [ ]:
PROBLEMS = [
    {"q": "A store sells apples for $0.50 each and oranges for $0.75 each. Sarah buys 8 apples and 6 oranges. How many cents does she spend in total?", "answer": "850"},
    {"q": "A factory produces 240 widgets per hour. If it runs 8 hours per day for 5 days, then has a 15% quality reject rate, how many good widgets are produced?", "answer": "8160"},
    {"q": "Train A leaves at 9am at 60 mph. Train B leaves the same station at 11am at 90 mph. At what hour does Train B catch Train A? (answer as whole hours after 9am)", "answer": "7"},
    {"q": "A rectangle has perimeter 56 cm. Its length is 3 times its width. What is its area in square cm?", "answer": "147"},
    {"q": "A tank holds 500 liters. It fills at 25 liters/min and drains at 10 liters/min. Starting empty, how many minutes to fill it?", "answer": "34"},
    {"q": "A class of 30 students: 40% scored above 80, 30% scored 60-80, the rest below 60. How many students scored below 60?", "answer": "9"},
    {"q": "A car travels 180 km in 2 hours, then 120 km in 1.5 hours. What is the average speed for the whole trip in km/h?", "answer": "84"},
    {"q": "A ladder is 10m long and rests against a wall. The base is 6m from the wall. How high up the wall does it reach in meters?", "answer": "8"},
    {"q": "John invests $1000 at 5% simple interest per year. How much interest does he earn over 3 years?", "answer": "150"},
    {"q": "There are 12 teams in a round-robin tournament where every team plays every other team exactly once. How many games are played total?", "answer": "66"},
]

greedy_correct = 0
sc_correct     = 0
sc_details     = []

for item in PROBLEMS:
    greedy_resp  = cot_answer(item["q"], system="Think step by step. End with 'Answer: <number>'.")
    greedy_ans   = extract_number(greedy_resp)
    greedy_ok    = greedy_ans == item["answer"]
    greedy_correct += int(greedy_ok)

    samples       = sample_cot(item["q"], n=10)
    sc_ans, votes = majority_vote(samples)
    sc_ok         = sc_ans == item["answer"]
    sc_correct    += int(sc_ok)

    sc_details.append({
        "q":         item["q"][:50] + "...",
        "gold":      item["answer"],
        "greedy":    greedy_ans,
        "greedy_ok": greedy_ok,
        "sc":        sc_ans,
        "sc_ok":     sc_ok,
        "votes":     votes,
    })

print(f"Greedy CoT:       {greedy_correct}/10")
print(f"Self-consistency: {sc_correct}/10")

Displaying results with vote distributions:

In [ ]:
#| code-fold: true
rows = []
for d in sc_details:
    rows.append({
        "Problem":            d["q"],
        "Gold":               d["gold"],
        "Greedy":             d["greedy"],
        "Greedy ✓":           "✓" if d["greedy_ok"] else "✗",
        "SC":                 d["sc"],
        "SC ✓":               "✓" if d["sc_ok"] else "✗",
        "Vote distribution":  str(d["votes"]),
    })

with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(rows))

The vote distribution column tells the story: on problems where self-consistency helps, the correct answer attracts a clear plurality while error answers scatter across multiple distinct wrong values. The failure regime is when the model has a systematic conceptual error — all samples converge to the same wrong answer, so the vote simply confirms the error. Self-consistency is diversity insurance; it cannot rescue a model from a concept it genuinely does not understand.

## Tree of Thoughts (Lite)

### From Chain to Tree

Self-consistency generates $N$ *complete* parallel paths and votes at the end — there is no pruning, no backtracking, and no evaluation of partial states. Tree of Thoughts (ToT) from Yao et al. (2023) introduces a richer structure: an explicit search tree where each node is a *partial solution state*, and a separate evaluator scores partial states before the search commits to them.

Two LLM roles work in concert: (1) a **thought generator** that proposes $k$ candidate next steps from any given state, and (2) a **state evaluator** that classifies each partial state as `sure`, `maybe`, or `impossible`. Two search strategies are available: **BFS** (fixed depth $d$, keep top-$b$ states per level) and **DFS** (explore to completion, backtrack on `impossible`).

The contrast with prior methods:

| Method | Paths | Evaluate partials? | Backtrack? |
|---|---|---|---|
| CoT (greedy) | 1 | No | No |
| Self-consistency | $N$ complete | No | No |
| ToT | $b^d$ partial | Yes | Yes |

The evaluator is the key ingredient. It converts an open-ended search into a tractable one by pruning clearly hopeless branches early, before they waste generator calls at deeper levels.

### The Game of 24

Game of 24 is a standard benchmark for combinatorial reasoning: use each of four given numbers exactly once with $+$, $-$, $\times$, $\div$ to reach $24$. Yao et al. (2023) report that greedy CoT with GPT-4 solves only about $4\%$ of hard instances. The task is well-suited to ToT because partial states (the remaining numbers after one operation) are easy to evaluate — a rule-based or LLM evaluator can quickly judge whether $24$ is still reachable.

We use a simplified **3-number variant** to keep the tree depth at $2$ — manageable for a demo without excessive API calls. In this variant, two arithmetic operations are applied in sequence: $\text{op}_1(a, b) = x$, then $\text{op}_2(x, c) = 24.$

## Code: ToT Solver

The thought generator proposes $k$ candidate single-operation steps from a set of numbers, and the state evaluator classifies each resulting partial state:

In [ ]:
# **Thought generator.**
PROPOSE_PROMPT = """\
You are solving a variant of Game of 24 using exactly 3 numbers.
Given a set of numbers, propose exactly {k} distinct ways to apply one arithmetic operation
(+, -, *, /) to two of the numbers, producing a result.
Format each proposal on a new line as:
  <num1> <op> <num2> = <result> (remaining: <leftover>)
Example for numbers [4, 6, 8]:
  4 + 6 = 10 (remaining: 8)
  8 - 6 = 2 (remaining: 4)
  4 * 6 = 24 (remaining: 8)\
"""

def propose_thoughts(numbers: list[int | float], k: int = 3) -> list[str]:
    h = ChatHistory()
    h.update(PROPOSE_PROMPT.format(k=k), role="system")
    h.update(f"Numbers: {numbers}. Propose {k} operations.", role="user")
    raw = comp.create(h, temperature=0.7)
    lines = [
        line.strip()
        for line in raw.strip().splitlines()
        if line.strip() and "=" in line
    ]
    return lines[:k]  # <1>


# **State evaluator.**
EVALUATE_PROMPT = """\
You are evaluating Game of 24 states.
Given remaining numbers, respond with exactly one word:
  sure       — you can clearly reach 24 from these numbers
  maybe      — might be possible, not immediately obvious
  impossible — provably cannot reach 24 from these numbers\
"""

def evaluate_state(remaining: list[int | float]) -> str:
    h = ChatHistory()
    h.update(EVALUATE_PROMPT, role="system")
    h.update(f"Remaining: {remaining}. Can these reach 24?", role="user")
    verdict = comp.create(h, temperature=0).strip().lower()  # <2>
    for token in ("sure", "impossible", "maybe"):
        if token in verdict:
            return token
    return "maybe"

1. Lines without `=` are discarded — they are likely preamble or formatting noise from the model.
2. Temperature $0$ for the evaluator: we want a consistent scoring signal, not diversity.

The BFS solver maintains a frontier of partial states, expands each with $k$ generator proposals, prunes `impossible` states, and keeps the top-$b$ survivors by evaluator verdict at each depth:

In [ ]:
# **BFS solver.**
def parse_result(thought_line: str) -> float | None:
    """Extract the numeric result of the applied operation from a thought line."""
    m = re.search(r"=\s*(-?[\d.]+)\s*\(remaining", thought_line)  # <3>
    return float(m.group(1)) if m else None

def solve_game24_tot(numbers: list[int], k: int = 3, beam_width: int = 2) -> list[dict]:
    """BFS over thought tree with beam search; return kept states at final depth."""
    frontier = [{"numbers": list(map(float, numbers)), "history": [], "verdict": "maybe"}]

    for depth in range(len(numbers) - 1):
        next_frontier = []
        for state in frontier:
            thoughts = propose_thoughts(state["numbers"], k=k)
            for thought in thoughts:
                result = parse_result(thought)
                if result is None:
                    continue
                remaining_nums = state["numbers"].copy()  # <4>
                rem_match = re.search(r"remaining:\s*([-\d.\s]+)\)", thought)
                if rem_match:
                    try:
                        remaining_nums = [
                            float(x)
                            for x in rem_match.group(1).split()
                            if x not in ("-", "")
                        ]
                    except ValueError:
                        remaining_nums = [result]
                elif len(state["numbers"]) > 1:
                    remaining_nums = [result]

                verdict = evaluate_state(remaining_nums)
                if verdict == "impossible":
                    continue
                next_frontier.append({
                    "numbers":  remaining_nums,
                    "history":  state["history"] + [thought],
                    "verdict":  verdict,
                })

        priority = {"sure": 0, "maybe": 1}
        next_frontier.sort(key=lambda s: priority.get(s.get("verdict", "maybe"), 1))
        frontier = next_frontier[:beam_width] if next_frontier else frontier  # <5>

    return frontier

3. We extract the numeric result from the expression to track what single number remains at each depth.
4. In the 3-number variant, each operation reduces the count by $1$: $[a, b, c] \to [\text{result}, c] \to [\text{final}].$
5. If no non-impossible states remain, the frontier is kept as-is to avoid an empty beam — graceful degradation.

### Comparison: Single-Shot vs. CoT vs. ToT

We define single-shot and zero-shot CoT baselines, then run all three approaches on a set of 3-number Game of 24 cases with known solutions:

In [ ]:
# **Single-shot.**
def single_shot_24(numbers: list[int]) -> str:
    h = ChatHistory()
    h.update("Solve Game of 24: use the given numbers with +, -, *, / to reach 24.", role="system")
    h.update(f"Numbers: {numbers}. Show your solution.", role="user")
    return comp.create(h, temperature=0)


# **Zero-shot CoT.**
def cot_24(numbers: list[int]) -> str:
    h = ChatHistory()
    h.update("Think step by step to make 24 from the given numbers using +, -, *, /.", role="system")
    h.update(f"Numbers: {numbers}.", role="user")
    return comp.create(h, temperature=0)


TEST_CASES = [
    {"numbers": [2, 3, 4], "solution": "(2*3)*4 = 24"},   # two multiplications
    {"numbers": [1, 4, 6], "solution": "4*6/1 = 24"},     # trivial — baseline sanity check
    {"numbers": [4, 5, 1], "solution": "4*(5+1) = 24"},   # add-then-multiply
    {"numbers": [3, 8, 9], "solution": "(9/3)*8 = 24"},   # divide-then-multiply, non-obvious
]

Running the comparison and displaying results:

In [ ]:
#| code-fold: true
def verify_24(expr: str) -> bool:
    try:
        return abs(eval(expr.split("=")[0].strip()) - 24) < 1e-6
    except (ValueError, ZeroDivisionError, SyntaxError, NameError, TypeError):
        return False

rows = []
for case in TEST_CASES:
    nums     = case["numbers"]
    single   = single_shot_24(nums)
    cot_resp = cot_24(nums)
    tot      = solve_game24_tot(nums, k=3, beam_width=2)
    tot_ans  = tot[-1]["history"] if tot else []

    rows.append({
        "Numbers":    str(nums),
        "Solution":   case["solution"],
        "Single-shot": single[:100].replace("\n", " "),
        "CoT":         cot_resp[:100].replace("\n", " "),
        "ToT steps":   " → ".join(tot_ans) if tot_ans else "—",
    })

with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(rows))

**Cost analysis.** ToT makes $k \times \text{depth} \times 2$ calls per puzzle (generator + evaluator at each node) versus $1$ for greedy CoT. For the 3-number variant with $k = 3$ and depth $= 2$, that is roughly $12$ calls compared to $1$. The overhead is worthwhile when: (a) the problem has verifiable partial states, (b) correctness matters more than latency, and (c) a cheap evaluator is available. The divison-then-multiply case $[3, 8, 9]$ is the key stress test: a greedy CoT pass tends to try obvious operations first ($8 + 9 = 17$, $9 - 3 = 6$) and miss the non-obvious $9 / 3 = 3 \to 3 \times 8 = 24$ path. ToT explores this by design.

## The Inference-Time Compute Landscape

### Test-Time Compute as a Knob

The classical way to improve a language model is to train a larger one. It turns out there is a second axis: allocating more *inference* compute — more tokens, more calls, more search — to a fixed model. Snell et al. (2024) show that a compute-optimal strategy combining a smaller model with $14\times$ inference compute can match a model $14\times$ larger at training time. This reframes the scaling conversation: quality is a joint function of (a) model capacity and (b) how hard we make the model think at inference.

The three strategies we have studied occupy distinct positions on this cost spectrum:

| Strategy | Test-time compute | Mechanism | Verification needed? |
|---|---|---|---|
| Greedy CoT | $O(L)$ | single chain | no |
| Self-consistency ($N$) | $O(NL)$ | ensemble + vote | no |
| ToT ($k, b, d$) | $O(kbdL)$ | tree search + eval | yes |

Two additional results round out the picture. Brown et al. (2024) "Large Language Monkeys" showed that *coverage* — the probability that at least one sample is correct — scales log-linearly with $N$, but *selecting* the correct one from many candidates requires an external verifier. This is the core bottleneck: generating good candidates is easier than identifying them. Budget forcing from Kimi k1.5 (2025) offers a low-overhead alternative: append a reconsideration prompt mid-generation, forcing the model to allocate additional tokens to the same problem serially.

### Implementing Budget Forcing

We implement budget forcing as a multi-turn conversation where the model's prior response is injected as an `assistant` turn, then a reconsideration phrase is added as a new `user` turn:

In [ ]:
# **Budget forcing.**
def budget_forced_cot(problem: str, n_reconsiderations: int = 1) -> str:
    """Force additional thinking by appending reconsideration turns."""
    h = ChatHistory()
    h.update(
        "Think step by step. Be careful and check your work. "
        "End with 'Answer: <number>'.",
        role="system",
    )
    h.update(problem, role="user")

    response = comp.create(h, temperature=0)
    h.update(response, role="assistant")  # <1>

    for _ in range(n_reconsiderations):
        h.update("Wait — let me reconsider that from scratch.", role="user")  # <2>
        response = comp.create(h, temperature=0)
        h.update(response, role="assistant")

    return response


forced   = budget_forced_cot(TRAP_PROBLEM, n_reconsiderations=2)
baseline = cot_answer(TRAP_PROBLEM)

print("=== Baseline ===")
print(baseline)
print("\n=== Budget-forced (2 reconsiderations) ===")
print(forced)

1. Adding the first response as an `assistant` turn lets the model see its own prior reasoning as context to reconsider.
2. The reconsideration phrase is injected as a new `user` turn — budget forcing via multi-turn conversation rather than logit manipulation.

Budget forcing often catches the planted error in `TRAP_PROBLEM` because the reconsideration pass re-reads the problem with the prior (erroneous) answer visible, sometimes triggering a self-correction. But it is not reliable — the model may simply reaffirm its prior answer with more confidence. Unlike self-consistency (parallel diversity) and ToT (structured search), budget forcing is serial and lacks a structural scaffold. It is best understood as a near-zero-cost hedge for latency-critical settings where one additional API call is acceptable.

## Summary

Putting it all together, the choice of reasoning strategy reduces to a question of what resources are available and what failure mode you are defending against:

| Situation | Strategy | Why |
|---|---|---|
| Arithmetic / MCQ; short extractable answer | Self-consistency ($N = 5$–$10$) | Cheap; no evaluator needed |
| Multi-step planning; partial states verifiable | ToT with BFS + evaluator | Errors caught before propagation |
| Latency-critical; one answer fast | Greedy CoT + skeptical system prompt | No extra calls |
| High-stakes; external verifier available | Best-of-$N$ + verifier | Direct correctness maximization |
| Frozen model; want more "thinking" cheaply | Budget forcing | Near-zero implementation cost |

A final observation worth making: recent reasoning models — OpenAI o1, DeepSeek-R1, Kimi k1.5 — have *internalized* CoT via reinforcement learning, generating hidden `<think>` traces that are functionally equivalent to our explicit chains but with the verification signal baked into training. The same structural insights apply: error propagation, premise sycophancy, and test-time scaling are properties of the underlying reasoning process, not of the prompting layer. The strategies in this notebook compose naturally with those models — self-consistency over o1 outputs still outperforms single o1 calls on the hardest problems.

---

■